In [1]:
!pip install -q transformers torch accelerate sentencepiece pandas tqdm

In [2]:
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#Cek GPU

import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


Load model dari gdrive

In [4]:
MODEL_PATH = "/content/drive/MyDrive/SKRIPSI MANTAP/indobertweet"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModel.from_pretrained(MODEL_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)
model.eval()

print("Model berhasil dimuat")
print("Device :", device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: /content/drive/MyDrive/SKRIPSI MANTAP/indobertweet
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model berhasil dimuat
Device : cuda


In [5]:
# Path folder di Google Drive
folder_path = '/content/drive/MyDrive/SKRIPSI MANTAP/data sosmed/.Process/Clean data'

# Path file
file_path = f'{folder_path}/yt_clean.csv'

# Membaca CSV
df = pd.read_csv(
    file_path,
    sep=',',  # gunakan ',' jika disimpan dengan to_csv() default
    engine='python'
)

# Cek nama kolom
print(df.columns)

# Ambil kolom comment
texts = df["yt_clean"]

print("Jumlah komentar:", len(texts))

Index(['yt_clean'], dtype='object')
Jumlah komentar: 95630


In [6]:
#parameter
MAX_LENGTH = 50
BATCH_SIZE = 32

In [ ]:
import numpy as np

output_path = "/content/yt_embed_bilstm.npy"

embedding_file = np.lib.format.open_memmap(
    output_path,
    mode="w+",
    dtype=np.float32,
    shape=(len(texts), MAX_LENGTH, 768)
)

In [ ]:
import gc
from tqdm import tqdm
import torch

start = 0

for i in tqdm(range(0, len(texts), BATCH_SIZE)):

    batch = texts.iloc[i:i+BATCH_SIZE].tolist()

    encoded = tokenizer(
        batch,
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    encoded = {k: v.to(device) for k, v in encoded.items()}

    with torch.no_grad():
        outputs = model(**encoded)

        embeddings = (
            outputs.last_hidden_state
            .cpu()
            .numpy()
            .astype(np.float32)
        )

    end = start + len(batch)

    embedding_file[start:end] = embeddings

    start = end

    del encoded
    del outputs
    del embeddings

    torch.cuda.empty_cache()
    gc.collect()

embedding_file.flush()

del embedding_file
gc.collect()

print(f"Embedding berhasil disimpan di {output_path}")

 28%|██▊       | 839/2989 [05:54<15:36,  2.30it/s]

In [ ]:
embedding_file.flush()

del embedding_file
gc.collect()

print("Embedding berhasil disimpan.")

Embedding berhasil disimpan.


In [ ]:
size = len(texts) * MAX_LENGTH * 768 * 4
print(size / 1024**3, "GB")

13.679981231689453 GB


In [ ]:
output_path = "/content/yt_embed_bilstm.dat"

In [ ]:
output_path = "/content/yt_embed_bilstm.npy"

In [ ]:
!df -h

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   61G   52G  55% /
tmpfs            64M     0   64M   0% /dev
shm             5.7G  4.0K  5.7G   1% /dev/shm
/dev/root       2.0G  1.3G  696M  65% /usr/sbin/docker-init
tmpfs           6.4G  216K  6.4G   1% /var/colab
/dev/sda1       119G   64G   55G  54% /kaggle/input
tmpfs           6.4G     0  6.4G   0% /proc/acpi
tmpfs           6.4G     0  6.4G   0% /proc/scsi
tmpfs           6.4G     0  6.4G   0% /sys/firmware
drive            15G   12G  3.1G  80% /content/drive


In [ ]:
output_path = "/content/yt_embed_bilstm.npy"

embedding_file = np.memmap(
    output_path,
    dtype=np.float32,
    mode="w+",
    shape=(len(texts), MAX_LENGTH, 768)
)

In [ ]:
output_path = "/content/yt_embed_bilstm.dat"

embedding_file = np.memmap(
    output_path,
    dtype=np.float32,
    mode="w+",
    shape=(len(texts), MAX_LENGTH, 768)
)

In [ ]:
import shutil

shutil.make_archive(
    "/content/drive/MyDrive/SKRIPSI MANTAP/data sosmed/Youtube/yt_embed_bilstm.npy",
    "zip",
    root_dir="/content",
    base_dir="yt_embed_bilstm.npy"
)

FileNotFoundError: [Errno 2] No such file or directory: '/content/yt_embed_bilstm.npy'

GA DIPAKE

In [ ]:
def extract_embeddings(texts):

    all_embeddings = []

    texts = texts.fillna("").astype(str).reset_index(drop=True)

    for i in tqdm(range(0, len(texts), BATCH_SIZE)):

        batch = texts.iloc[i:i+BATCH_SIZE].astype(str).tolist()

        encoded = tokenizer(
            batch,
            padding="max_length",
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        encoded = {
            k: v.to(device)
            for k, v in encoded.items()
        }

        with torch.no_grad():
            outputs = model(**encoded)

        embeddings = outputs.last_hidden_state.cpu().numpy()

        all_embeddings.append(embeddings)

    return np.concatenate(all_embeddings, axis=0)

In [ ]:
X_embedding = extract_embeddings(df["yt_clean"])

  0%|          | 0/2989 [00:00<?, ?it/s]

In [ ]:
print(X_embedding.shape)

In [ ]:
np.save(
    "/content/drive/MyDrive/SKRIPSI MANTAP/data sosmed/Youtube/yt_embed_bilstm.npy",
    X_embedding
)